# Part A: levers 1 and 3 on the published LettuceDetect-large

No training. The published model (58.9 span F1 in its paper) predicts the RAGTruth test set; then
lever 1 (tuned span decoding) and lever 3 (character-level stacker) are fitted on one half of the test
set and scored on the other half, and the other way round (the model was trained on all of RAGTruth
train, so there is no other unseen data). About 15-20 minutes on a T4 GPU.

Choose **Runtime > Change runtime type > T4 GPU** first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
PROJECT = '/content/drive/MyDrive/hallu_poc'   # the folder with poc.py
WORK = f'{PROJECT}/work'                        # results are saved here
DATA = f'{PROJECT}/RAGTruth'
if not os.path.exists(f'{DATA}/response.jsonl') and not os.path.exists(f'{DATA}/dataset/response.jsonl'):
    DATA = '/content/RAGTruth'                   # not on Drive: get it from GitHub
    if not os.path.exists(DATA):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/ParticleMedia/RAGTruth', DATA], check=True)
print('data:', DATA)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# ModernBERT needs transformers 4.48 or newer
import transformers
from packaging import version
if version.parse(transformers.__version__) < version.parse('4.48'):
    !pip -q install -U 'transformers>=4.48'
    print('Updated transformers: Runtime > Restart session, then run the cells again.')
else:
    print('transformers', transformers.__version__)

## 1. Predict the test set

In [ ]:
!cd {PROJECT} && python -u poc.py predict --data-dir {DATA} --work-dir {WORK} \
    --model KRLabsOrg/lettucedect-large-modernbert-en-v1 --name ld_large

## 2. Levers 1 and 3 (2-fold cross-fitting on the test set)

In [ ]:
!cd {PROJECT} && python -u poc.py levers --data-dir {DATA} --work-dir {WORK} --name ld_large --fit crossfit

**Reading the table.** *baseline* is LettuceDetect's own decoding; compare its span F1 with the paper's 58.9.
*Gain vs baseline* is the paired difference on the same test answers, with a 95% bootstrap interval over
test sources: an interval that does not include 0 is a real gain. Results are saved in `work/results`.